# Credit-Constrained Model: When Targeting Can Actually Help

**Buera, Kaboski & Shin (2011)-style collateral constraint with heterogeneous net worth**

The base model showed that targeting productive firms still lowers output (vs. doing nothing).
This extension adds a **real, pre-existing friction**: the collateral constraint k ≤ λ·a, where
wealth and talent are uncorrelated. Now talented entrants can be stuck below efficient scale
simply because they're poor — exactly the kind of friction that targeted policy can correct.

This notebook solves four economies and compares aggregate output:
1. **Frictionless**: No capital constraint (upper-bound benchmark)
2. **Constrained**: Finite collateral multiplier, no policy
3. **Targeted**: Credit access favors top-quintile entrants
4. **Mistargeted**: Credit access favors bottom-quintile entrants

## Setup & Imports

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# Add src to path (go up one directory from notebooks/)
sys.path.insert(0, '../src')

from ge_model import (
    run_credit_scenario,
    CALIBRATION_CREDIT,
    s_grid_credit,
    plot_credit_output,
    plot_credit_constrained_share,
)

print("[OK] Imports successful")
print(f"[OK] Working directory: {os.getcwd()}")

## Model Roadmap: The Four-Part Structure

This model extends the base framework by introducing a **collateral constraint**. Here's how the pieces fit together:

### Part A: Entry & Firm Formation
Potential entrepreneurs arrive with two attributes drawn *independently at entry*:
- **Productivity (s)**: managerial talent, drawn from AR(1) process
- **Net worth (a)**: initial wealth, drawn from a separate distribution

The independence of s and a is the key friction: talented people aren't necessarily rich, and rich people aren't necessarily talented.

### Part B: Production & Capital/Labor Allocation
Each period, an active firm chooses how much capital (k) and labor (n) to deploy. Without constraints, this is straightforward Cobb-Douglas. **With the collateral constraint**, talented but poor firms hit a ceiling: they can borrow only up to k = λ·a.

### Part C: Exit & Dynamic Decisions
Firms compare the value of continuing against the cost of fixed operations. This endogenous exit rule shapes the steady-state population of firms.

### Part D: Market Equilibrium
Free entry ensures expected value equals entry cost; labor supply is exogenous. The equilibrium price clears the market for capital goods and labor.

### Policy Knob: Collateral Multiplier (λ)
The model varies λ across regimes, either uniformly (no policy) or targeting specific entry cohorts (by net worth quintile). This asks: **Can a government improve outcomes by easing credit for the right firms?**

## Part A: Entry & Firm Formation

### Calibration Parameters

The model balances two sources of heterogeneity: productivity and wealth. Both are persistent but uncorrelated at entry.

In [ ]:
# Display calibration
calib_display = {
    'Parameter': [
        'Productivity grid points',
        'AR(1) persistence (ρ)',
        'Innovation std dev (σ)',
        'Capital share (θ_k)',
        'Labor share (θ_n)',
        'Returns to scale',
        'Capital rental rate (r)',
        'Discount factor (β)',
        'Fixed operating cost',
        'Entry cost',
        'Aggregate labor supply',
        'Base collateral multiplier (λ_base)',
        'Boosted multiplier (λ_boost)',
    ],
    'Value': [
        CALIBRATION_CREDIT['N_Z'],
        CALIBRATION_CREDIT['RHO'],
        CALIBRATION_CREDIT['SIGMA'],
        CALIBRATION_CREDIT['THETA_K'],
        CALIBRATION_CREDIT['THETA_N'],
        f"{CALIBRATION_CREDIT['THETA_K'] + CALIBRATION_CREDIT['THETA_N']:.2f}",
        CALIBRATION_CREDIT['R_RATE'],
        CALIBRATION_CREDIT['BETA'],
        f"{CALIBRATION_CREDIT['CF']:.2f}",
        f"{CALIBRATION_CREDIT['CE']:.2f}",
        CALIBRATION_CREDIT['L_SUPPLY'],
        CALIBRATION_CREDIT['LAMBDA_BASE'],
        CALIBRATION_CREDIT['LAMBDA_BOOST'],
    ]
}
calib_df = pd.DataFrame(calib_display)
print("Calibration (stylized, illustrative):")
print(calib_df.to_string(index=False))
print(f"\nNet worth types (a): {CALIBRATION_CREDIT['A_VALS']}")
print("  → Wealth is drawn independently of productivity")
print("  → This independence IS the market failure we're trying to correct")

## Part B: Production & Capital/Labor Allocation

### The Firm's Problem Each Period

An active firm with productivity s and net worth a chooses capital k and labor n to maximize profit, subject to a collateral constraint that limits how much it can borrow.

In [ ]:
production_spec = """
PRODUCTION & ALLOCATION PROBLEM:

  max_{k,n}  π(s, k, n) = p · s · k^θ_k · n^θ_n - r·k - w·n
  
  subject to:  k ≤ λ · a    [collateral constraint]
  
INTERPRETATION:

  • p · s · k^θ_k · n^θ_n = output (Cobb-Douglas production)
    - s = firm's productivity (managerial talent)
    - λ ∈ [0.3, 0.5] = collateral multiplier (how much a firm can borrow per unit of net worth)
  
  • r·k = rental on capital (r ≈ 0.10, includes interest + depreciation)
  • w·n = wage bill (normalized w = 1.0)
  
  • θ_k = 0.30 (capital's share of output)
  • θ_n = 0.35 (labor's share of output)
  • Returns to scale = θ_k + θ_n = 0.65 < 1  (decreasing returns from above)

UNCONSTRAINED SOLUTION (if collateral constraint slack):

  k_unconstrained = [θ_k · p · s / r]^(1/(1-θ_k-θ_n)) · n^(θ_n/(1-θ_k-θ_n))
  n_unconstrained = [θ_n · p · s / w]^(1/(1-θ_k-θ_n)) · k^(θ_k/(1-θ_k-θ_n))

BINDING CONSTRAINT CASE (firm is collateral-constrained):

  If k_unconstrained > λ·a:
    • Set k = λ·a  (firm hits the borrowing ceiling)
    • Re-solve for n from labor FOC:
        n_constrained = [θ_n · p · s · (λ·a)^θ_k / w]^(1/(1-θ_n))
    • Firm produces below unconstrained scale

KEY INSIGHT:
  
  A high-productivity firm (large s) may still hit the constraint if it has low net worth (small a).
  Conversely, a low-productivity firm with high a will not want to borrow (and doesn't).
  → This creates genuine misallocation: talent and capital are mismatched.
"""

print(production_spec)

## Part C: Exit & Dynamic Decisions

### The Firm's Lifetime Value Function

Each firm compares the value of continuing to operate against the fixed cost. This endogenous exit rule determines which firms remain active in equilibrium.

In [ ]:
exit_spec = """
VALUE FUNCTION (Bellman equation, for a firm with wealth a and initial collateral multiplier λ):

  V(s | a, λ) = π(s, k(s), n(s)) - cf + β · E_s'[ max(V(s' | a, λ), 0) ]
  
  where:
    • π(s, k(s), n(s)) = static profit, given optimal choices under constraint
    • cf = 0.55 = fixed operating cost (must pay each period to stay active)
    • β = 0.85 = discount factor (firms care about future, but not too much)
    • E_s'[ · ] = expectation over next-period productivity
    • max(V(s'), 0) = exit option (if V < 0, firm exits, gets value 0)

STEADY-STATE EXIT RULE:
  
  Each firm has an endogenous "exit threshold" s*_exit(a, λ):
    • If current s ≤ s*_exit: firm exits
    • If current s > s*_exit: firm continues
  
  The exit threshold is higher for:
    - Poorer firms (low a) → need higher s to justify fixed cost
    - Tighter constraints (low λ) → harder to cover fixed cost

INTUITION:

  A talented but poor firm (high s, low a) may be severely constrained on capital.
  If the constraint is tight enough, even with high talent, net profit falls short of fixed cost.
  → It exits, wasting its talent.
  
  A weak but rich firm (low s, high a) can borrow freely, so even low profits cover fixed cost.
  → It survives, occupying resources that could go to better uses.
"""

print(exit_spec)

## Part D: Market Equilibrium

### Equilibrium Conditions

Equilibrium pins down the price of output (which is both a consumption good and capital good). Three conditions must hold:

In [ ]:
eq_spec = """
EQUILIBRIUM CONDITIONS:

1. FREE ENTRY (expected value at entry = entry cost):
  
    E_{s,a}[ max(V(s | a, λ), 0) ] = ce
    
    where:
      • E_{s,a} = expectation over both productivity AND net worth at entry
      • ce = 1.10 = entry cost (fee to start a firm, paid once)
    
    Interpretation:
      On average, a new entrant expects to recover their entry fee, but no more.
      If V > ce, entry surges → price falls → V falls back to ce.
      If V < ce, entry stops → price rises → V rises back to ce.

2. LABOR-MARKET CLEARING (aggregate labor demand = aggregate labor supply):
  
    ∫ n(s, a) · M(s, a) ds da = L_supply = 1.0
    
    where:
      • n(s, a) = labor demand of a firm with productivity s and wealth a
      • M(s, a) = steady-state mass (number) of firms of type (s, a)
    
    Interpretation:
      Exogenous labor supply (1.0 unit, normalized) must equal the sum of what firms demand.
      With one hour worked = one unit of output ≈ fixed, output differences are pure TFP.

3. CAPITAL MARKET CLEARING:
  
    (Implicit in free entry; equilibrium price p* ensures it clears.)

SOLUTION METHOD:

  1. Discretize productivity (s) via Tauchen (N=25 grid points)
  2. Discretize net worth (a) directly (N=5 values, for simplicity)
  3. For a candidate price p, solve the value function V(s|a,λ) via value function iteration
  4. Use Brentq root-finding to find p* satisfying free entry
  5. Compute steady-state mass M(s, a) from ergodic distribution
  6. Verify labor-market clearing (usually holds by construction)
"""

print(eq_spec)

## Numerical Implementation: Discretization & Calibration

### Discretizing the State Space

We approximate the continuous distributions of productivity and net worth using discrete grids. This trades off accuracy for computational speed.

In [ ]:
from ge_model import tauchen

# Discretize the AR(1) productivity process
z_grid, Q = tauchen(CALIBRATION_CREDIT['N_Z'], CALIBRATION_CREDIT['RHO'], CALIBRATION_CREDIT['SIGMA'])
s_grid_local = np.exp(z_grid)

# Plot the discretized process
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))

# Left: log-productivity grid
ax1.scatter(range(len(z_grid)), z_grid, alpha=0.6, s=50)
ax1.set_xlabel('Grid index')
ax1.set_ylabel('log-productivity (z)')
ax1.set_title(f'Productivity discretization (Tauchen): N={CALIBRATION_CREDIT["N_Z"]}, ρ={CALIBRATION_CREDIT["RHO"]}')
ax1.grid(alpha=0.3)

# Right: productivity grid (s = exp(z))
ax2.scatter(range(len(s_grid_local)), s_grid_local, alpha=0.6, s=50, color='#E69F00')
ax2.set_xlabel('Grid index')
ax2.set_ylabel('Productivity (s = exp(z))')
ax2.set_title('Exponentiated grid (firms\' production efficiency)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Productivity range: [{s_grid_local[0]:.3f}, {s_grid_local[-1]:.3f}]")
print(f"Net worth values: {CALIBRATION_CREDIT['A_VALS']}")

## Solving the Model: Four Policy Regimes

### The Policy Question

We solve four versions of the economy, varying the **collateral multiplier (λ)**—which governs how much a firm can borrow relative to its net worth:

1. **Frictionless** (λ = ∞): No borrowing limit → upper-bound benchmark
2. **Constrained** (λ = λ_base ≈ 0.3): Tight borrowing constraint, no policy
3. **Targeted** (λ varies by a): Top-quintile firms get higher λ
4. **Mistargeted** (λ varies by a): Bottom-quintile firms get higher λ

In [ ]:
# Run all four regimes
regimes_credit = ["frictionless", "constrained", "targeted", "mistargeted"]
results_credit = {}

for regime in regimes_credit:
    print(f"Solving {regime}...", end=" ")
    res = run_credit_scenario(regime)
    results_credit[regime] = res
    print(f"✓ p*={res['p']:.4f}, Y={res['aggregate_output']:.4f}, constrained={100*res['constrained_share']:.1f}%")

print("\n✓ All regimes solved.")

## Results & Findings

### Equilibrium Results Across Regimes

In [ ]:
# Build results table
rows = []
Y_frictionless = results_credit["frictionless"]["aggregate_output"]
Y_constrained = results_credit["constrained"]["aggregate_output"]

for regime in regimes_credit:
    res = results_credit[regime]
    gap_closed = 100 * (res["aggregate_output"] - Y_constrained) / (Y_frictionless - Y_constrained)
    
    rows.append({
        "Regime": regime.capitalize(),
        "Output": f"{res['aggregate_output']:.4f}",
        "% of frictionless": f"{100 * res['aggregate_output'] / Y_frictionless:.1f}%",
        "Gap closed (%)": f"{gap_closed:.1f}%",
        "Constrained firms (%)": f"{100 * res['constrained_share']:.1f}%",
        "Exit rate": f"{res['exit_rate']:.3f}",
    })

table_credit = pd.DataFrame(rows)
print("\nEQUILIBRIUM RESULTS:")
print(table_credit.to_string(index=False))

# Save to CSV
os.makedirs('../results/credit', exist_ok=True)
table_credit.to_csv('../results/credit/results_table.csv', index=False)
print("\n[OK] Saved: ../results/credit/results_table.csv")

### Visualizations

In [ ]:
# Chart 1: Output Levels
labels_credit = {
    "frictionless": "Frictionless\n(no credit constraint)",
    "constrained": "Constrained, no policy\n(wealth uncorrelated w/ talent)",
    "targeted": "Targeted credit access\n(top-quintile entrants favored)",
    "mistargeted": "Mistargeted credit access\n(bottom-quintile entrants favored)",
}

fig, ax = plot_credit_output(results_credit, regimes_credit, labels_credit, save_path='../results/credit/output.png')
plt.show()
print("[OK] Saved: ../results/credit/output.png")

In [ ]:
# Chart 2: Constrained Share
fig, ax = plot_credit_constrained_share(results_credit, regimes_credit, labels_credit, save_path='../results/credit/constrained_share.png')
plt.show()
print("[OK] Saved: ../results/credit/constrained_share.png")

### Finding 1: Targeted Access Works (When There's a Real Friction)

In [ ]:
finding1 = f"""
TARGETED CREDIT ACCESS RECOVERS OUTPUT LOSS

  Frictionless output:              {Y_frictionless:.4f}
  Constrained output (no policy):   {Y_constrained:.4f}
  Targeted policy output:           {results_credit['targeted']['aggregate_output']:.4f}
  
  Friction gap closed by targeting: {100 * (results_credit['targeted']['aggregate_output'] - Y_constrained) / (Y_frictionless - Y_constrained):.1f}%
  
  → Targeted credit access recovers about half the loss from the collateral constraint.
  → This is fundamentally DIFFERENT from the base model, where targeting always hurts.
  → The difference: here, the friction is REAL (wealth ≠ talent), not a wedge.
"""

print(finding1)

### Finding 2: Mistargeting Is Wasted, Not Harmful

In [ ]:
finding2 = f"""
MISTARGETING WASTES RESOURCES BUT DOESN'T DESTROY OUTPUT

  Constrained output (no policy):  {Y_constrained:.4f}
  Mistargeted policy output:       {results_credit['mistargeted']['aggregate_output']:.4f}
  Difference:                      {results_credit['mistargeted']['aggregate_output'] - Y_constrained:.4f}
  
  WHY THE DIFFERENCE FROM THE BASE MODEL?
  
    • Base model (pure tax/subsidy): forcing a distortion always harms output
      Even on strong firms, a tax/subsidy creates a wedge that misallocates capital.
    
    • Credit model (optional credit): relaxing collateral is just an OPTION
      A weak firm (low productivity) will not borrow extra capital it has no productive use for.
      Mistargeting wastes the policy but doesn't destroy output.
  
  KEY INSIGHT: You can't force a firm to borrow for purposes it doesn't have.
"""

print(finding2)

### Finding 3: The Meta-Lesson — When Does Targeting Work?

In [ ]:
finding3 = """
WHEN DOES "PICK THE WINNERS" ACTUALLY IMPROVE OUTCOMES?

The answer depends entirely on the SOURCE OF INEFFICIENCY.

1. PURE WEDGES (distortionary taxes, subsidies, regulations):
   ✗ Targeting is ALWAYS bad
   → Even high-productivity firms experience output loss from the wedge
   → No targeting can overcome this (Base Model result)

2. MARKET FAILURES with independent state variables (credit constraints, externalities):
   ✓ Targeting CAN be good—but only if:
      a) The friction is REAL (not just a wedge)
      b) The state variable (talent, s) is INDEPENDENT of the source of friction (wealth, a)
      c) The policy precisely targets the constraint, not the distortion
   
3. ASYMMETRIC INFORMATION (firms hide talent from lenders):
   ? Targeting MIGHT help by credibly signaling to markets
   → But easy to get wrong; signaling can backfire

CONCLUSION FOR POLICYMAKERS:

Before deploying "winner-picking" interventions, diagnose:
  • Is there a real market failure (not just a wedge)?
  • Are the binding constraints independent of firm productivity?
  • Can you accurately identify the beneficiary population?
  
Get these right → targeting works.
Get them wrong → targeting wastes resources (at best) or actively harms (at worst).
"""

print(finding3)

### Constraint Intensity Across Regimes

In [ ]:
print("COLLATERAL CONSTRAINT INTENSITY (share of firms constrained):")
print()
for regime in regimes_credit:
    share = 100 * results_credit[regime]['constrained_share']
    print(f"  {regime:15s}: {share:6.1f}% of firms operate at capital limit")
print()
print("Interpretation:")
print("  • 'Frictionless' shows near-zero (firms choose capital freely)")
print("  • 'Constrained' shows the baseline friction intensity")
print("  • 'Targeted' reduces constraint intensity for top firms (they get higher λ)")
print("  • 'Mistargeted' still has high intensity (low-wealth firms stay constrained)")